# Harmony360 Media Engine v0.1 — Colab Notebook

This notebook bootstraps the Harmony360-native media pipeline in Google Colab.

**Hard mandate:** every Harmony360-owned executable media class inherits from the retained `Harmony360` core class. The core file is copied verbatim from the governed recovery lineage and is not rewritten by this notebook.

v0.1 is intentionally conservative:
- human approval is required before upload preparation;
- YouTube upload preparation is **private-only**;
- generated visuals are not evidence;
- a PASS result is never automatically described as scientific proof.


In [ ]:
# CELL 1 — Mount Google Drive
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

MASTER = Path('/content/drive/MyDrive/HAR360_MASTER')
MEDIA_ROOT = MASTER / 'media' / 'youtube' / 'harmony360_building_the_framework'
MEDIA_ROOT.mkdir(parents=True, exist_ok=True)
print('MEDIA_ROOT:', MEDIA_ROOT)


In [ ]:
# CELL 2 — Clone / update GitHub project
import subprocess, pathlib

REPO_URL = 'https://github.com/rdntmsn/Harmony360.git'
WORK = pathlib.Path('/content/Harmony360')
if WORK.exists():
    subprocess.run(['git', '-C', str(WORK), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(WORK)], check=True)

PROJECT = WORK / 'media_engine'
print('PROJECT:', PROJECT)
assert PROJECT.exists(), 'media_engine folder not found in GitHub checkout'


In [ ]:
# CELL 3 — Install project + test dependency
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'pytest>=8'], check=True)
print('Installed Harmony360 Media Engine v0.1')


In [ ]:
# CELL 4 — Verify retained core SHA-256 before use
from pathlib import Path
import hashlib, json

core_path = PROJECT / 'harmony360_media' / 'core' / 'harmony360_core.py'
prov = json.loads((PROJECT / 'source_provenance.json').read_text())

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

print('runtime core:', sha256(core_path))
print('expected    :', prov['core_selection']['sha256'])
assert sha256(core_path) == prov['core_selection']['sha256']
print('✅ Core provenance verified')


In [ ]:
# CELL 5 — Run inheritance and pipeline tests
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(PROJECT / 'tests')], check=True)
print('✅ Tests passed')


In [ ]:
# CELL 6 — Create Episode 001 governed package
from harmony360_media.orchestrator import MediaOrchestrator

RUN_DIR = MEDIA_ROOT / 'EP001'
RUN_DIR.mkdir(parents=True, exist_ok=True)

orch = MediaOrchestrator(memory_dir=RUN_DIR / 'har360')

episode = orch.create_episode(
    episode_id='H360-YT-EP001',
    series='Harmony360: Building the Framework',
    topic='Why PASS Does Not Mean Proof',
    sources=[
        {
            'source_id': 'H360-CORE-V1-SPEC',
            'title': 'Harmony360 Core v1.0 Specification',
            'source_type': 'governed_document',
            'classification': 'UNDER_REVIEW',
        }
    ],
    claims=[
        {
            'claim_id': 'CLAIM-001',
            'statement': 'A PASS result means the defined test criteria were satisfied.',
            'claim_class': 'implemented',
            'status': 'SUPPORTED',
            'evidence_ids': ['H360-CORE-V1-SPEC'],
        },
        {
            'claim_id': 'CLAIM-002',
            'statement': 'A PASS result does not automatically establish the larger scientific claim.',
            'claim_class': 'implemented',
            'status': 'SUPPORTED',
            'evidence_ids': ['H360-CORE-V1-SPEC'],
        },
    ],
    script=(
        'What if an experiment succeeds and still proves nothing? '
        'In Harmony360, a PASS result means the defined test criteria were satisfied. '
        'It does not automatically establish the larger scientific claim. '
        'The result and the claim remain separate, and the next question is what evidence '
        'would be required to trust the claim more.'
    ),
    scenes=[
        {
            'scene_id': 'S001', 'order': 1,
            'narration': 'What if an experiment succeeds and still proves nothing?',
            'visual_prompt': 'dark scientific space, luminous geometric question mark, Harmony360 visual language',
            'duration_seconds': 6.0,
        },
        {
            'scene_id': 'S002', 'order': 2,
            'narration': 'In Harmony360, a PASS result means the defined test criteria were satisfied.',
            'visual_prompt': 'evidence ledger showing PASS inside a bounded test box',
            'duration_seconds': 8.0,
        },
        {
            'scene_id': 'S003', 'order': 3,
            'narration': 'It does not automatically establish the larger scientific claim.',
            'visual_prompt': 'PASS box separated from larger CLAIM node by a guarded boundary',
            'duration_seconds': 8.0,
        },
        {
            'scene_id': 'S004', 'order': 4,
            'narration': 'The result and the claim remain separate.',
            'visual_prompt': 'two luminous nodes labeled RESULT and CLAIM connected by evidence lineage',
            'duration_seconds': 7.0,
        },
        {
            'scene_id': 'S005', 'order': 5,
            'narration': 'The next question is what evidence would be required to trust the claim more.',
            'visual_prompt': 'branching evidence path toward reconstruction and human review',
            'duration_seconds': 9.0,
        },
    ],
    title='Why a PASS Result Still Does Not Mean Proof',
    description='Harmony360: Building the Framework — Episode 1. A governed explanation of why successful test execution and scientific proof are not the same thing.'
)

print('workflow_state:', episode.workflow_state)
print('claim_gate:', episode.metadata['claim_gate'])


In [ ]:
# CELL 7 — Write captions + governed episode ledger
from harmony360_media.agents import CaptionAgent, MediaLedger
import json

CaptionAgent(memory_dir=RUN_DIR / 'har360').write_srt(episode, RUN_DIR / 'captions.srt')
ledger_path = MediaLedger(memory_dir=RUN_DIR / 'har360').write_episode(episode, RUN_DIR)

print('Episode JSON:', ledger_path)
print('HAR360 files:', list((RUN_DIR / 'har360').glob('*.har360')))


In [ ]:
# CELL 8 — Human review gate
from harmony360_media.agents import PublicationGate

print(episode.script)
print('\nClaim gate:', episode.metadata.get('claim_gate'))
print('\nCurrent state:', episode.workflow_state)

# Do NOT change this to APPROVE until you personally review the script and assets.
HUMAN_DECISION = 'HOLD'  # APPROVE / REJECT / HOLD / REGENERATE_SCRIPT / REGENERATE_VISUALS / RENDER_AGAIN

PublicationGate(memory_dir=RUN_DIR / 'har360').record_review(
    episode,
    reviewer='human_governance',
    decision=HUMAN_DECISION,
    notes='Colab v0.1 review gate'
)
print('Review decision:', episode.review)


In [ ]:
# CELL 9 — Prepare YouTube upload only after approval
from harmony360_media.agents import YouTubeAdapter

yt = YouTubeAdapter(memory_dir=RUN_DIR / 'har360')
if episode.review and episode.review.decision == 'APPROVE':
    upload_payload = yt.prepare_private_upload(episode)
    print(json.dumps(upload_payload, indent=2))
    print('✅ Upload payload prepared. v0.1 remains private-only.')
else:
    print('⛔ Upload preparation blocked until HUMAN_DECISION == APPROVE')


## Next build stage

The next controlled increment is to add provider adapters for:
1. image generation / approved stock or local visuals;
2. narration/TTS;
3. music bed mixing;
4. FFmpeg render;
5. YouTube OAuth + `videos.insert` upload as **private**;
6. publication receipt written back into the episode `.har360` lineage.

Those adapters remain downstream of the same Harmony360 core, claim gate, and human publication gate.
